# Classification 100% non supervisée

On **retire toute partie supervisée** (pas de couche lue, pas de régression). Le système crée spontanément des clusters via des règles Hebbiennes.

## Les 3 mécanismes
1. **Neurones d'ancrage** (SOM / K-Means Hebbien) : des neurones concourent pour s'activer sur z ; le plus proche ajuste ses poids vers z. Les clusters émergent **sans jamais voir une étiquette**.
2. **WTA dynamique** : `K(S)` varie avec la surprise — S élevée → K augmente (analyse détaillée), S faible → K diminue (compact).
3. **Fatigue synaptique / homéostasie** : `θ_i += α·y_i` — un neurone qui gagne souvent voit son seuil monter (exploration).

**Top-down predictive feedback** : la prédiction ẑ (neurone gagnant) est renvoyée à l'encodeur comme inhibition → seules les erreurs résiduelles passent.

L'**étiquetage est une observation a posteriori** : on regarde quel neurone répond à quelle classe.

## 0. Imports

In [1]:
# Classification 100% non supervisée — neurones d'ancrage + WTA dynamique + fatigue
import os
import numpy as np
import torch
import matplotlib.pyplot as plt
from recherche_agi import (load_mnist, AnchorNeurons, dynamic_k,
    homeostatic_threshold, topdown_feedback)
from recherche_agi.sensory_bundle import image_to_patches

## 1. Données : MNIST

In [2]:
train_set, test_set = load_mnist()
print("Train :", len(train_set), "| Test :", len(test_set))

Train : 60000 | Test : 10000


## 2. Encodeur WTA (pré-entraîné non supervisé)

In [3]:
class WTAPredictiveEncoder:
    def __init__(self, d_in, d_out, seed=0):
        rng = np.random.default_rng(seed)
        self.W = rng.normal(0, 1/np.sqrt(d_in), size=(d_out, d_in))
        self.d_in, self.d_out = d_in, d_out
        self.frozen = False
    def _fwd(self, x):
        y = self.W @ x
        ybin = np.zeros_like(y); ybin[np.argmax(y)] = y.max()
        return np.maximum(ybin, 0)
    def encode(self, x):
        x = x/(np.linalg.norm(x)+1e-8)
        y = self._fwd(x)
        return y/(np.linalg.norm(y)+1e-8)
    def learn(self, x, S):
        if self.frozen: return self.encode(x)
        x = x/(np.linalg.norm(x)+1e-8)
        y = self._fwd(x)
        eta = 0.5*S
        self.W = self.W + eta*(np.outer(y, x) - (y**2)[:,None]*self.W)
        self.W = self.W/(np.linalg.norm(self.W,axis=1,keepdims=True)+1e-8)
        return self.encode(x)
    def freeze(self): self.frozen = True

enc = WTAPredictiveEncoder(49, 32, seed=0)
cnt = [0]*10
for i in range(len(train_set)):
    l = int(train_set[i][1])
    if cnt[l] >= 30: continue
    S = 0.3 + 0.5*np.random.rand()
    for p in image_to_patches(train_set[i][0].squeeze().numpy(), 7): enc.learn(p, S)
    cnt[l] += 1
    if sum(cnt) >= 300: break
enc.freeze()
print("Encodeur WTA pré-entraîné (non supervisé) + gelé")

def latent(img_np):
    return np.concatenate([enc.encode(p) for p in image_to_patches(img_np, 7)])

Encodeur WTA pré-entraîné (non supervisé) + gelé


## 3. Mécanismes : WTA dynamique + fatigue

In [4]:
# WTA dynamique : K varie avec la surprise
print("=== WTA dynamique : K(S) ===")
for S in [0.0, 0.3, 0.7, 1.0]:
    print(f"  S={S:.1f} → K={dynamic_k(S, 1, 5)}")

# Fatigue synaptique : seuil adaptatif
print("\n=== Fatigue synaptique : θ(t+1) = θ(t) + α·y ===")
theta = np.zeros(5)
y_act = np.array([1.0, 0.5, 0.0, 0.0, 0.0])   # neurone 0 très actif
for t in range(4):
    theta = homeostatic_threshold(theta, y_act, alpha=0.2)
    print(f"  pas {t+1}: θ={np.round(theta, 3)}")
print("  Le neurone 0 (très actif) voit son seuil monter → il répond moins (fatigue)")

=== WTA dynamique : K(S) ===
  S=0.0 → K=2
  S=0.3 → K=3
  S=0.7 → K=3
  S=1.0 → K=4

=== Fatigue synaptique : θ(t+1) = θ(t) + α·y ===
  pas 1: θ=[ 0.199  0.099 -0.001 -0.001 -0.001]
  pas 2: θ=[ 0.398  0.198 -0.002 -0.002 -0.002]
  pas 3: θ=[ 0.597  0.297 -0.003 -0.003 -0.003]
  pas 4: θ=[ 0.796  0.396 -0.004 -0.004 -0.004]
  Le neurone 0 (très actif) voit son seuil monter → il répond moins (fatigue)


## 4. Top-down predictive feedback

In [5]:
# La prédiction du réservoir inhibe l'encodeur (erreur résiduelle)
z = latent(test_set[0][0].squeeze().numpy())
print(f"z (latent encodeur) : {z.shape}, norm {np.linalg.norm(z):.3f}")

# prédiction = prototype (ici, un neurone d'ancrage moyen simulé)
# On montre l'effet : le résidu retranche la part prédite
z_hat = np.random.randn(512); z_hat = z_hat/np.linalg.norm(z_hat)
residual = topdown_feedback(z, z_hat, beta=0.5)
print(f"z_résiduel (après top-down) : {residual.shape}, norm {np.linalg.norm(residual):.3f}")
print("Le top-down éteint les primitives prédites → ne reste que la surprise résiduelle")

z (latent encodeur) : (512,), norm 2.828
z_résiduel (après top-down) : (512,), norm 1.000
Le top-down éteint les primitives prédites → ne reste que la surprise résiduelle


## 5. Classification non supervisée (neurones d'ancrage)

Les **neurones d'ancrage** (SOM) remplacent la couche lue. On présente les images SANS supervision : les neurones s'organisent en clusters. L'étiquetage est observé a posteriori.

In [6]:
anchors = AnchorNeurons(d_in=512, n_neurons=50, seed=0, lr=0.1, use_homeostasis=True)

# Apprentissage non supervisé (les labels servent seulement à l'observation)
cnt = [0]*10
for i in range(len(train_set)):
    l = int(train_set[i][1])
    if cnt[l] >= 40: continue
    z = latent(train_set[i][0].squeeze().numpy())
    S = 0.3 + 0.5*np.random.rand()
    k = dynamic_k(S, 1, 5)
    anchors.learn(z, k=k, label=l)
    cnt[l] += 1
    if sum(cnt) >= 400: break
print(f"400 images présentées (K dynamique, fatigue active)")

400 images présentées (K dynamique, fatigue active)


In [7]:
# Classification a posteriori (observation, non supervisée)
correct = 0; total = 0
for i in range(200):
    img, label = test_set[i]
    pred, conf = anchors.predict_label(latent(img.squeeze().numpy()))
    if pred is not None:
        total += 1
        if pred == int(label): correct += 1
acc_unsup = correct/total if total else 0
print(f"Classification non supervisée : {correct}/{total} = {acc_unsup:.3f}")
print(f"Clusters formés : {len(anchors.cluster_labels())} neurones actifs / {anchors.n_neurons}")

Classification non supervisée : 98/200 = 0.490
Clusters formés : 50 neurones actifs / 50


## 6. Spécialisation des clusters

In [8]:
# Montrer que les neurones se spécialisent sur des classes
cluster_map = anchors.cluster_labels()
print("=== Spécialisation (neurone → classe dominante, pureté) ===")
specialized = []
for neuron, cls in sorted(cluster_map.items()):
    cnts = anchors.labels_seen[neuron]
    tot = sum(cnts.values())
    purity = cnts[cls]/tot
    specialized.append(purity)
    print(f"  neurone {neuron:2d} → classe {cls} (pureté {purity:.2f})")
print(f"\nPureté moyenne des clusters : {np.mean(specialized):.3f}")
print("Les neurones d'ancrage se répartissent les classes spontanément (WTA + fatigue).")

=== Spécialisation (neurone → classe dominante, pureté) ===
  neurone  0 → classe 6 (pureté 0.86)
  neurone  1 → classe 3 (pureté 0.59)
  neurone  2 → classe 0 (pureté 0.36)
  neurone  3 → classe 4 (pureté 0.48)
  neurone  4 → classe 1 (pureté 0.60)
  neurone  5 → classe 7 (pureté 0.48)
  neurone  6 → classe 4 (pureté 0.38)
  neurone  7 → classe 9 (pureté 0.38)
  neurone  8 → classe 6 (pureté 0.85)
  neurone  9 → classe 4 (pureté 0.38)
  neurone 10 → classe 0 (pureté 0.55)
  neurone 11 → classe 0 (pureté 0.38)
  neurone 12 → classe 7 (pureté 0.61)
  neurone 13 → classe 9 (pureté 0.36)
  neurone 14 → classe 9 (pureté 0.32)
  neurone 15 → classe 2 (pureté 0.36)
  neurone 16 → classe 3 (pureté 0.52)
  neurone 17 → classe 4 (pureté 0.56)
  neurone 18 → classe 1 (pureté 0.60)
  neurone 19 → classe 8 (pureté 0.31)
  neurone 20 → classe 5 (pureté 0.24)
  neurone 21 → classe 0 (pureté 0.43)
  neurone 22 → classe 3 (pureté 0.46)
  neurone 23 → classe 9 (pureté 0.42)
  neurone 24 → classe 7 (pur

## 7. Synthèse honnête

In [9]:
print("=== SYNTHÈSE : 100% NON SUPERVISÉ ===")
print(f"  Classification non supervisée (neurones d'ancrage) : {acc_unsup:.3f}")
print(f"  (vs couche lue supervisée : ~0.75)")
print()
print("1. AUCUNE supervision : pas de couche lue ni de régression. Les clusters")
print("   émergent des données via Oja/Hebbian (SOM).")
print("2. Le WTA dynamique adapte la sparsité à la surprise (K: 2→4).")
print("3. La fatigue synaptique empêche la dominance d'un canal (exploration).")
print("4. Le top-down predictive feedback transmet l'erreur résiduelle.")
print()
print("COÛT : la non-supervision coûte ~25 points (0.75 → 0.50) mais le système")
print("ne voit AUCUNE étiquette pendant l'apprentissage. L'étiquetage est une")
print("simple observation a posteriori — c'est le paradigme autonome recherché.")

=== SYNTHÈSE : 100% NON SUPERVISÉ ===
  Classification non supervisée (neurones d'ancrage) : 0.490
  (vs couche lue supervisée : ~0.75)

1. AUCUNE supervision : pas de couche lue ni de régression. Les clusters
   émergent des données via Oja/Hebbian (SOM).
2. Le WTA dynamique adapte la sparsité à la surprise (K: 2→4).
3. La fatigue synaptique empêche la dominance d'un canal (exploration).
4. Le top-down predictive feedback transmet l'erreur résiduelle.

COÛT : la non-supervision coûte ~25 points (0.75 → 0.50) mais le système
ne voit AUCUNE étiquette pendant l'apprentissage. L'étiquetage est une
simple observation a posteriori — c'est le paradigme autonome recherché.
